In [1]:
import boto3
import sagemaker
from sagemaker import Session
from sagemaker.transformer import Transformer
from botocore.exceptions import ClientError


s3_client = boto3.client('s3', region_name="eu-north-1")

def find_model_path(bucket_name, prefix):
    try:
        response = s3_client.list_objects_v2(Bucket=bucket_name, Prefix=prefix)
        for obj in response.get('Contents', []):
            if obj['Key'].endswith('model.tar.gz'):
                return f"s3://{bucket_name}/{obj['Key']}"
        raise FileNotFoundError(f"No model.tar.gz found under prefix {prefix}")
    except ClientError as e:
        print(f"Error listing objects in S3: {e}")
        raise

# SageMaker and boto3 settings
region = "eu-north-1"
boto_session = boto3.Session(region_name=region)
sagemaker_client = boto3.client("sagemaker", region_name=region)
sagemaker_session = Session(boto_session=boto_session)
role = sagemaker.get_execution_role()
image_uri = sagemaker.image_uris.retrieve("kmeans", region=region)


k_values = range(6, 7)

for k in k_values:
    print(f"Running batch inference for k={k}...")

    prefix = f"kmeans/k_{k}/"
    model_path = find_model_path(bucket_name="bdp-models", prefix=prefix)

    print(f"Model data URL for k={k}: {model_path}")

    # Define output path
    output_path = f"s3://bdp-inference-results/kmeans/k_{k}/"

    # Register the SageMaker model
    model_name = f"kmeans-k-{k}"
    try:
        sagemaker_client.create_model(
            ModelName=model_name,
            PrimaryContainer={
                "Image": image_uri,
                "ModelDataUrl": model_path,
                "Environment": {},
            },
            ExecutionRoleArn=role,
        )
        print(f"Model {model_name} registered successfully.")
    except sagemaker_client.exceptions.ClientError as e:
        print(f"Error registering model {model_name}: {str(e)}")
        continue

    # Create a Transformer for batch inference
    transformer = Transformer(
        model_name="kmeans-k-6",
        instance_count=1,
        instance_type="ml.c5.2xlarge",
        strategy="MultiRecord",
        output_path=output_path,
        assemble_with="Line",
        accept="text/csv",
        sagemaker_session=sagemaker_session,
        input_filter="$[1:]",
        output_filter="$[0,-2,-1]" 
    )

    # Start the batch transform job
    try:
        transformer.transform(
            data='s3://bdp-test-data/scaled/test-data.csv',
            content_type="text/csv",
            split_type="Line",
            join_source="Input"
        )
        print(f"Inference for k={k} completed. Results saved to: {output_path}")
    except Exception as e:
        print(f"Error during inference for k={k}: {str(e)}")


/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/pydantic/_internal/_fields.py:192: UserWarning: Field name "json" in "MonitoringDatasetFormat" shadows an attribute in parent "Base"
  warnings.warn(


[01/24/25 16:55:26] INFO     Found credentials from IAM Role:                                   ]8;id=580850;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=355709;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/botocore/credentials.py#1075\1075]8;;\
                             BaseNotebookInstanceEc2InstanceRole                                                   

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


[01/24/25 16:55:30] INFO     Found credentials from IAM Role:                                   ]8;id=735281;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=225806;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/botocore/credentials.py#1075\1075]8;;\
                             BaseNotebookInstanceEc2InstanceRole                                                   

                    INFO     Found credentials from IAM Role:                                   ]8;id=233629;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=672198;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/botocore/credentials.py#1075\1075]8;;\
                             BaseNotebookInstanceEc2InstanceRole                                                   

                    INFO     Same images used for training and inference. Defaulting to image     ]8;id=483594;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/image_uris.py\image_uris.py]8;;\:]8;id=138965;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/image_uris.py#391\391]8;;\
                             scope: inference.                                                                     

                    INFO     Ignoring unnecessary instance type: None.                            ]8;id=443075;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/image_uris.py\image_uris.py]8;;\:]8;id=3198;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/image_uris.py#528\528]8;;\

Running batch inference for k=6...
Model data URL for k=6: s3://bdp-models/kmeans/k_6/kmeans-2025-01-22-18-29-22-012/output/model.tar.gz
Error registering model kmeans-k-6: An error occurred (ValidationException) when calling the CreateModel operation: Cannot create already existing model "arn:aws:sagemaker:eu-north-1:982534349340:model/kmeans-k-6".
